# 1. 데이터셋

In [2]:
import pandas as pd
import numpy as np

# 1. CSV 파일 로드 (dtype 옵션으로 기업코드를 문자열로 직접 지정)
file_path_long = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv'
file_path_2025 = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_DART_ESG_보고서공시_2025.csv'

df_long = pd.read_csv(file_path_long, dtype={'기업코드': str})
df_2025 = pd.read_csv(file_path_2025, dtype={'기업코드': str})

# 2. 기업코드 포맷 통일 (앞자리 0이 누락된 경우 6자리로 채움)
join_key = '기업코드' 
df_long[join_key] = df_long[join_key].astype(str).str.zfill(6)
df_2025[join_key] = df_2025[join_key].astype(str).str.zfill(6)

# 3. 병합 설정 및 중복 제거
target_cols = ['지속가능경영보고서_발행', '기업지배구조보고서_발행', '보고서_발행_유형']
df_2025_subset = df_2025[[join_key] + target_cols].drop_duplicates(subset=[join_key])

# 4. Left Join 수행
merged_df = pd.merge(df_long, df_2025_subset, on=join_key, how='left')

# 5. 평가년도가 2025가 아닌 행은 해당 컬럼들을 NaN 처리
is_not_2025 = merged_df['평가년도'].astype(str) != '2025'
merged_df.loc[is_not_2025, target_cols] = np.nan

# 6. 최종 데이터 CSV 저장
output_filename = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long_merged.csv'
merged_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"조인이 완료되어 '{output_filename}' 파일로 저장되었습니다.")

조인이 완료되어 '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long_merged.csv' 파일로 저장되었습니다.


In [3]:
import pandas as pd
import numpy as np

# 1. CSV 파일 로드
file_path_long = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2021-2025_Long.csv'
file_path_2025 = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_DART_ESG_보고서공시_2025.csv'

df_long = pd.read_csv(file_path_long, dtype={'기업코드': str})
df_2025 = pd.read_csv(file_path_2025, dtype={'기업코드': str})

# 2. 기업코드 포맷 통일 (6자리 채움)
join_key = '기업코드' 
df_long[join_key] = df_long[join_key].astype(str).str.zfill(6)
df_2025[join_key] = df_2025[join_key].astype(str).str.zfill(6)

# 3. 병합 설정 및 중복 제거
target_cols = ['지속가능경영보고서_발행', '기업지배구조보고서_발행', '보고서_발행_유형']
df_2025_subset = df_2025[[join_key] + target_cols].drop_duplicates(subset=[join_key])

# 4. Left Join 수행
merged_df = pd.merge(df_long, df_2025_subset, on=join_key, how='left')

# 5. 평가년도가 2025가 아닌 행은 해당 컬럼들을 NaN 처리
is_not_2025 = merged_df['평가년도'].astype(str) != '2025'
merged_df.loc[is_not_2025, target_cols] = np.nan

# 6. 평가년도 2021 데이터 삭제 (2022~2025년 데이터만 남김)
merged_df = merged_df[merged_df['평가년도'].astype(str) != '2021'].copy()

# 7. 최종 데이터 CSV 저장 (파일명 변경)
output_filename = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'
merged_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"조인 및 2021년 데이터 삭제가 완료되어 '{output_filename}' 파일로 저장되었습니다.")

조인 및 2021년 데이터 삭제가 완료되어 '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv' 파일로 저장되었습니다.


In [5]:
import io
import time
import zipfile
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import xml.etree.ElementTree as ET

# ==============================================================================
# 1. DART API KEY 및 파일 경로 설정
# ==============================================================================
DART_API_KEY = "4c056faa738bcb25aa545b05a5b1d69137a71f39" 

file_path = "/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv"
output_path = "/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged_v2.csv"

# ==============================================================================
# 2. SSL/네트워크 오류 자동 재시도를 위한 Session 생성 함수
# ==============================================================================
def create_robust_session():
    session = requests.Session()
    # 브라우저 요청으로 위장하기 위한 User-Agent 설정
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    })
    # 연결 실패 또는 특정 HTTP 에러 시 최대 5회 자동 재시도 (지연시간 자동 증가)
    retries = Retry(
        total=5,
        backoff_factor=1,  # 재시도 간격: 1초, 2초, 4초 ...
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    return session

session = create_robust_session()

# ==============================================================================
# 3. DART 고유번호(corp_code) 매핑 테이블 다운로드
# ==============================================================================
def get_corp_code_map(api_key):
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    params = {"crtfc_key": api_key}
    res = session.get(url, params=params, timeout=15)

    corp_map = {}
    with zipfile.ZipFile(io.BytesIO(res.content)) as z:
        with z.open("CORPCODE.xml") as f:
            tree = ET.parse(f)
            root = tree.getroot()
            for list_item in root.findall("list"):
                stock_code = list_item.findtext("stock_code")
                corp_code = list_item.findtext("corp_code")
                if stock_code and stock_code.strip():
                    corp_map[stock_code.strip().zfill(6)] = corp_code.strip()
    return corp_map

# ==============================================================================
# 4. 메인 실행 프로세스
# ==============================================================================
df = pd.read_csv(file_path, dtype={"기업코드": str})
df["기업코드"] = df["기업코드"].str.zfill(6)

print("DART 기업 고유번호 매핑 정보 수집 중...")
corp_map = get_corp_code_map(DART_API_KEY)

target_years = [2022, 2023, 2024]
unique_codes = df[df["평가년도"].isin(target_years)]["기업코드"].unique()
disclosure_results = {}

print(f"2022~2024년 DART 공시 조회 시작 (대상 기업 수: {len(unique_codes)}개)...")

for idx, code in enumerate(unique_codes, 1):
    corp_code = corp_map.get(code)
    if not corp_code:
        continue

    for year in target_years:
        start_date = f"{year}0101"
        end_date = f"{year}1231"

        url = "https://opendart.fss.or.kr/api/list.json"
        params = {
            "crtfc_key": DART_API_KEY,
            "corp_code": corp_code,
            "bde_dt": start_date,
            "pblntf_ty": "I",
            "page_count": "100",
        }

        has_sustainability = 0.0
        has_governance = 0.0

        try:
            # session.get 사용 및 타임아웃 10초 설정
            res_response = session.get(url, params=params, timeout=10)
            res = res_response.json()
            
            if res.get("status") == "000" and "list" in res:
                for report in res["list"]:
                    title = report.get("report_nm", "")
                    if "지속가능경영" in title or "ESG" in title:
                        has_sustainability = 1.0
                    if "지배구조" in title:
                        has_governance = 1.0

        except Exception as e:
            print(f"오류 발생 ({code}, {year}년): {e}")

        # 요청 간 간격을 0.1초로 늘려 서버 차단 방지
        time.sleep(0.1)

        if has_sustainability == 1.0 and has_governance == 1.0:
            report_type = "둘 다 발행"
        elif has_sustainability == 1.0 and has_governance == 0.0:
            report_type = "지속가능경영만"
        elif has_sustainability == 0.0 and has_governance == 1.0:
            report_type = "지배구조만"
        else:
            report_type = "미발행"

        disclosure_results[(code, year)] = {
            "지속가능경영보고서_발행": has_sustainability,
            "기업지배구조보고서_발행": has_governance,
            "보고서_발행_유형": report_type,
        }

    if idx % 50 == 0 or idx == len(unique_codes):
        print(f"진행 상황: {idx}/{len(unique_codes)} 기업 완료")

# ==============================================================================
# 5. 결과 데이터 조인 및 저장
# ==============================================================================
for (code, year), values in disclosure_results.items():
    mask = (df["기업코드"] == code) & (df["평가년도"] == year)
    df.loc[mask, "지속가능경영보고서_발행"] = values["지속가능경영보고서_발행"]
    df.loc[mask, "기업지배구조보고서_발행"] = values["기업지배구조보고서_발행"]
    df.loc[mask, "보고서_발행_유형"] = values["보고서_발행_유형"]

df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"작업 완료! 결과가 다음 경로에 저장되었습니다:\n{output_path}")

DART 기업 고유번호 매핑 정보 수집 중...
2022~2024년 DART 공시 조회 시작 (대상 기업 수: 1123개)...
진행 상황: 50/1123 기업 완료
진행 상황: 100/1123 기업 완료
진행 상황: 150/1123 기업 완료
진행 상황: 200/1123 기업 완료
진행 상황: 250/1123 기업 완료
진행 상황: 300/1123 기업 완료
진행 상황: 350/1123 기업 완료
진행 상황: 400/1123 기업 완료
진행 상황: 450/1123 기업 완료
진행 상황: 500/1123 기업 완료
진행 상황: 550/1123 기업 완료
진행 상황: 600/1123 기업 완료
진행 상황: 650/1123 기업 완료
진행 상황: 700/1123 기업 완료
진행 상황: 750/1123 기업 완료
진행 상황: 800/1123 기업 완료
진행 상황: 850/1123 기업 완료
진행 상황: 900/1123 기업 완료
진행 상황: 950/1123 기업 완료
진행 상황: 1000/1123 기업 완료
진행 상황: 1050/1123 기업 완료
진행 상황: 1100/1123 기업 완료
진행 상황: 1123/1123 기업 완료
작업 완료! 결과가 다음 경로에 저장되었습니다:
/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged_v2.csv


In [6]:
import pandas as pd
import numpy as np

# 1. 파일 경로 설정
file_v2_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged_v2.csv'
file_v1_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'
output_path = '/Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv'

# 2. CSV 파일 로드 (기업코드는 6자리 문자열 타입 유지)
df_v2 = pd.read_csv(file_v2_path, dtype={'기업코드': str})
df_v1 = pd.read_csv(file_v1_path, dtype={'기업코드': str})

df_v2['기업코드'] = df_v2['기업코드'].astype(str).str.zfill(6)
df_v1['기업코드'] = df_v1['기업코드'].astype(str).str.zfill(6)

target_cols = ['지속가능경영보고서_발행', '기업지배구조보고서_발행', '보고서_발행_유형']

# 3. v1 파일에서 2025년 데이터의 기업코드 및 대상 컬럼 3개 추출
df_v1_2025 = df_v1[df_v1['평가년도'].astype(str) == '2025'][['기업코드'] + target_cols].drop_duplicates(subset=['기업코드'])

# 4. v2 파일과 v1_2025 데이터 Left Join 수행
df_merged = pd.merge(df_v2, df_v1_2025, on='기업코드', how='left', suffixes=('', '_v1'))

# 5. 평가년도가 2025인 행에 대해서만 v1 데이터 값으로 복원/업데이트
mask_2025 = df_merged['평가년도'].astype(str) == '2025'
for col in target_cols:
    df_merged.loc[mask_2025, col] = df_merged.loc[mask_2025, col + '_v1']

# 임시 컬럼 삭제
df_merged.drop(columns=[col + '_v1' for col in target_cols], inplace=True)

# 6. (선택사항) 2025년 데이터 중 공시가 없는 기업(NaN)을 0.0 및 '미발행'으로 채우고 싶을 경우
# *(확실하지 않은 정보 표시: 원본 v1에서도 공시 정보가 없던 762개 기업은 NaN으로 존재합니다. 0.0과 '미발행'으로 채우길 원하시면 아래 주석을 해제하세요)*
# df_merged.loc[mask_2025, '지속가능경영보고서_발행'] = df_merged.loc[mask_2025, '지속가능경영보고서_발행'].fillna(0.0)
# df_merged.loc[mask_2025, '기업지배구조보고서_발행'] = df_merged.loc[mask_2025, '기업지배구조보고서_발행'].fillna(0.0)
# df_merged.loc[mask_2025, '보고서_발행_유형'] = df_merged.loc[mask_2025, '보고서_발행_유형'].fillna('미발행')

# 7. 최종 CSV 파일 저장
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"2025년 데이터 복원이 완료되었습니다.\n저장 경로: {output_path}")

2025년 데이터 복원이 완료되었습니다.
저장 경로: /Users/lusia/Personal/13. DA Bootcamp/04. Project/2. ESG/01_데이터셋/KCGS_평가등급_2022-2025_Long_merged.csv
